# Interactive Web Map: Senior Access to Essential Services in Uusimaa
Building a multi-layer interactive map displaying 65+ senior population distribution along with health facilities, pharmacies, and supermarket nodes across Uusimaa.

In [ ]:
import pathlib
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

# Set workspace paths
data_dir = pathlib.Path("./data")
output_dir = pathlib.Path("./docs")
output_dir.mkdir(exist_ok=True)

# Read spatial layers
grid = gpd.read_file(data_dir / "uusimaa_demographics.gpkg")
health = gpd.read_file(data_dir / "healthcare_facilities.gpkg")
pharmacies = gpd.read_file(data_dir / "pharmacies.gpkg")
supermarkets = gpd.read_file(data_dir / "supermarkets.gpkg")

# Reproject to WGS84 for Folium compatibility
grid_wgs84 = grid.to_crs(epsg=4326)
health_wgs84 = health.to_crs(epsg=4326)
pharmacies_wgs84 = pharmacies.to_crs(epsg=4326)
supermarkets_wgs84 = supermarkets.to_crs(epsg=4326)

# Initialize interactive base map centered on Uusimaa region
m = folium.Map(location=[60.2500, 24.9000], zoom_start=9, tiles="CartoDB positron")

# Overlay demographic grid
folium.GeoJson(
    grid_wgs84,
    name="Senior Population Grid",
    style_function=lambda x: {
        "fillColor": "#fd8d3c",
        "color": "#ffffff",
        "weight": 0.3,
        "fillOpacity": 0.4,
    }
).add_to(m)

# Add clustered service layers
health_cluster = MarkerCluster(name="Healthcare Centers").add_to(m)
for idx, row in health_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup="Healthcare Center",
        icon=folium.Icon(color="red", icon="plus")
    ).add_to(health_cluster)

pharmacy_cluster = MarkerCluster(name="Pharmacies").add_to(m)
for idx, row in pharmacies_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup="Pharmacy",
        icon=folium.Icon(color="green", icon="info-sign")
    ).add_to(pharmacy_cluster)

supermarket_cluster = MarkerCluster(name="Supermarkets").add_to(m)
for idx, row in supermarkets_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup="Supermarket",
        icon=folium.Icon(color="blue", icon="shopping-cart")
    ).add_to(supermarket_cluster)

folium.LayerControl().add_to(m)

# Export interactive HTML web map
output_html = output_dir / "senior_services_interactive_map.html"
m.save(output_html)
print(f"Map successfully exported to: {output_html}")
